# PageRank: The Random Surfer Model

## Learning Objectives

1. **Describe** the random surfer model and its connection to PageRank
2. **Construct** the transition matrix for a small web graph
3. **Explain** why PageRank equals the stationary distribution of the random walk
4. **Identify** the conditions required for convergence (Perron-Frobenius)

## The Web as a Graph

The Web can be modelled as a **directed graph**:
- **Nodes:** web pages
- **Edges:** hyperlinks (page $u$ links to page $v$)

**Key question:** which pages are most important?

**PageRank insight (Brin & Page, 1998):** a page is important if many important pages link to it.
This is a recursive definition — solved by modelling a random surfer walking the web.

## The Random Surfer

Imagine a user who:
1. Starts at a random page
2. At each step, clicks a random outgoing link
3. Repeats indefinitely

After many steps, the fraction of time spent on each page converges to the **stationary distribution** of this random walk.

**PageRank of page $v$** = the long-run fraction of time the random surfer spends on $v$.

**Intuition:** pages with many incoming links from important pages will be visited more often.

In [ ]:
from collections import defaultdict

def build_transition_matrix(edges, nodes):
    """Build column-stochastic transition matrix for random surfer.
    Column j represents: if currently on page j, probability of going to each page.
    """
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    out_deg = defaultdict(int)
    for u,v in edges: out_deg[u] += 1
    M = [[0.0]*n for _ in range(n)]
    for u,v in edges:
        if out_deg[u] > 0:
            M[idx[v]][idx[u]] = 1.0 / out_deg[u]
    return M

# Small 4-page web
nodes = ['A','B','C','D']
edges = [('A','B'),('A','C'),('B','D'),('C','B'),('C','D'),('D','A')]
M = build_transition_matrix(edges, nodes)

print("Transition matrix (column j = from page j):")
print(f"{'':>8}" + "".join(f"{n:>8}" for n in nodes))
for i,row in enumerate(M):
    print(f"{nodes[i]:>8}" + "".join(f"{v:>8.3f}" for v in row))

## Stationary Distribution

A probability vector $\mathbf{r}$ is a **stationary distribution** if:

$$M \mathbf{r} = \mathbf{r}$$

For a **column-stochastic** matrix $M$ (each column sums to 1):
- At least one stationary distribution always exists (Perron-Frobenius)
- It is unique iff the Markov chain is **irreducible** (any state reachable from any other) and **aperiodic** (no periodic cycles)

**Power iteration** finds $\mathbf{r}$: start with $\mathbf{r}_0 = \mathbf{1}/n$, repeatedly multiply by $M$.

In [ ]:
def power_iteration(M, n_iter=50):
    n = len(M)
    r = [1.0/n] * n
    for _ in range(n_iter):
        new_r = [0.0]*n
        for i in range(n):
            for j in range(n):
                new_r[i] += M[i][j] * r[j]
        r = new_r
    return r

r = power_iteration(M)
print("PageRank (stationary distribution):")
for node, rank in zip(nodes, r):
    bar = '#' * int(rank * 40)
    print(f"  {node}: {rank:.4f}  {bar}")
print(f"
Sum of ranks: {sum(r):.4f} (should be 1.0)")

## Perron-Frobenius Conditions

For the random surfer to converge to a **unique** stationary distribution:

1. **Irreducibility:** the graph must be strongly connected (every page reachable from every page)
2. **Aperiodicity:** the graph must not have purely periodic structure

Real web graphs violate both:
- **Dead ends:** pages with no outgoing links
- **Spider traps:** groups of pages that link only to each other
- **Disconnected components:** some pages unreachable from others

**Fix:** teleportation (covered in the next notebook).

## Summary

| Concept | Definition | Role in PageRank |
|---------|-----------|-----------------|
| Random surfer | User clicking random links | Motivation for PageRank |
| Transition matrix | $M_{ij}$ = P(go to $i$ \| at $j$) | The Markov chain |
| Stationary distribution | $M\mathbf{r}=\mathbf{r}$ | PageRank vector |
| Perron-Frobenius | Conditions for unique convergence | Why teleportation is needed |

PageRank = stationary distribution of random walk on the web graph.
Without modifications (teleportation), the random surfer gets stuck in traps or wanders forever.